In [37]:
!pip -q install transformers datasets peft trl accelerate bitsandbytes



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [38]:
import os
import json
import torch

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    BitsAndBytesConfig
)
from peft import LoraConfig
from trl import SFTTrainer


In [39]:
# W&B is optional. If you don't have an account, just skip logging.
os.environ["WANDB_DISABLED"] = "true"


In [40]:
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
import os

BASE_DIR = os.getcwd()  # should be C:\Users\273441\Downloads\AIPlatform
TRAIN_FILE = os.path.join(BASE_DIR,  "brand_voice_train.jsonl")
EVAL_FILE = os.path.join(BASE_DIR,  "brand_voice_eval.jsonl")
OUTPUT_DIR = os.path.join(BASE_DIR,  "task4_outputs")


In [41]:
train_ds = load_dataset("json", data_files=TRAIN_FILE, split="train")
eval_ds = load_dataset("json", data_files=EVAL_FILE, split="train")

print(train_ds[0])
print("Train size:", len(train_ds))
print("Eval size:", len(eval_ds))

Generating train split: 15 examples [00:00, 282.39 examples/s]
Generating train split: 5 examples [00:00, 131.65 examples/s]

{'text': "### Instruction:\nRewrite the support response in NOVA's brand voice.\n\n### Input:\nYour order has been shipped and should arrive by Feb 14.\n\n### Response:\nGreat news — your order is already on the way and should reach you by Feb 14. You’re almost there ✨"}
Train size: 15
Eval size: 5


In [42]:
# Ensure dataset has a single 'text' field for SFTTrainer
def format_example(example):
    # Expected fields in JSONL: 'prompt' and 'response'
    prompt = example.get('prompt', '').strip()
    response = example.get('response', '').strip()
    return {"text": f"{prompt}\n{response}"}

train_ds = train_ds.map(format_example, remove_columns=train_ds.column_names)
eval_ds = eval_ds.map(format_example, remove_columns=eval_ds.column_names)
print(train_ds[0])


Map:   0%|          | 0/15 [00:00<?, ? examples/s]

Map: 100%|██████████| 5/5 [00:00<00:00, 168.72 examples/s]

{'text': '\n'}


In [43]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [44]:
use_4bit = torch.cuda.is_available()

bnb_config = BitsAndBytesConfig(
    load_in_4bit=use_4bit,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)


In [45]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config if use_4bit else None,
    device_map="auto" if use_4bit else None
)

if not use_4bit:
    model.to("cpu")

model.config.use_cache = False

# Avoid max_length vs max_new_tokens warning
model.generation_config.max_length = None


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 1530.53it/s]


In [46]:
peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"]
)

In [47]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    learning_rate=2e-4,
    logging_steps=5,
    eval_steps=10,
    save_steps=10,
    eval_strategy="steps",
    save_strategy="steps",
    report_to="none",
    run_name="nova-brand-voice-qlora",
    fp16=torch.cuda.is_available(),
    optim="paged_adamw_8bit" if torch.cuda.is_available() else "adamw_torch"
)


In [48]:
trainer = SFTTrainer(
    model=model,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    peft_config=peft_config,
    args=training_args
)

Truncating eval dataset: 100%|██████████| 5/5 [00:00<00:00, 196.72 examples/s]


In [49]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.
c:\Users\273441\Downloads\AIPlatform\env\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss,Validation Loss
6,2.211461,1.311168


TrainOutput(global_step=6, training_loss=2.072727163632711, metrics={'train_runtime': 92.6811, 'train_samples_per_second': 0.486, 'train_steps_per_second': 0.065, 'total_flos': 1119706398720.0, 'train_loss': 2.072727163632711})

In [50]:
trainer.model.save_pretrained(os.path.join(OUTPUT_DIR, "nova_brand_voice_adapter"))
tokenizer.save_pretrained(os.path.join(OUTPUT_DIR, "nova_brand_voice_adapter"))

print("Saved adapter to:", os.path.join(OUTPUT_DIR, "nova_brand_voice_adapter"))

Saved adapter to: c:\Users\273441\Downloads\AIPlatform\task4\task4_outputs\nova_brand_voice_adapter


In [51]:
def generate_response(prompt, max_new_tokens=120):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    # Return only the generated continuation (not the prompt)
    gen_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(gen_tokens, skip_special_tokens=True).strip()


In [52]:
eval_samples = [
    "Rewrite in NOVA brand voice: Your order has been shipped and will arrive by Feb 14.",
    "Rewrite in NOVA brand voice: Please share your order ID.",
    "Rewrite in NOVA brand voice: This issue will be escalated to a human agent.",
    "Rewrite in NOVA brand voice: This product is suitable for dry skin."
]

for s in eval_samples:
    print("\n" + "=" * 100)
    print("PROMPT:", s)
    print("OUTPUT:")
    print(generate_response(s, max_new_tokens=80))


PROMPT: Rewrite in NOVA brand voice: Your order has been shipped and will arrive by Feb 14.
OUTPUT:
2. Use a call-to-action (CTA) to encourage customers to take action: Click here to place your order now and receive a 10% discount.

3. Use a clear and concise headline: Shop now and save 10% on your order.

4. Use a strong and memorable subheading:

PROMPT: Rewrite in NOVA brand voice: Please share your order ID.
OUTPUT:
2. Email:
Subject: Order #12345

Dear [Customer Name],

We are delighted to inform you that your order #12345 has been successfully placed. We appreciate your business and look forward to serving you in the future.

If you have any questions or concerns, please do not hesitate to contact us at

PROMPT: Rewrite in NOVA brand voice: This issue will be escalated to a human agent.
OUTPUT:
2. Customer Service:

- Customer Service:

- Customer Service:

- Customer Service:

- Customer Service:

- Customer Service:

- Customer Service:

- Customer Service:

- Customer Service

In [53]:
results = []

for s in eval_samples:
    out = generate_response(s, max_new_tokens=None)
    results.append({
        "prompt": s,
        "output": out
    })

with open(os.path.join(OUTPUT_DIR, "task4_eval_outputs.json"), "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print("Saved evaluation outputs.")

c:\Users\273441\Downloads\AIPlatform\env\Lib\site-packages\transformers\generation\utils.py:1554: UserWarning: Using the model-agnostic default `max_length` (=43) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(
c:\Users\273441\Downloads\AIPlatform\env\Lib\site-packages\transformers\generation\utils.py:1554: UserWarning: Using the model-agnostic default `max_length` (=34) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(
c:\Users\273441\Downloads\AIPlatform\env\Lib\site-packages\transformers\generation\utils.py:1554: UserWarning: Using the model-agnostic default `max_length` (=40) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(
c:\Users\273441\Downloads\AIPlatform\env\Lib\site-packages\transformers\generation\utils.py:1554:

Saved evaluation outputs.
